# Notebook 5 — Final Model and Evaluation

Train the final Gradient Boosting model with balanced feature contribution.

## Configuration
- Model: Gradient Boosting Classifier
- max_features=3 (forces use of features beyond dist_to_major_river)
- Threshold: 0.4 (optimal F1)
- Train: 2019-2023 monsoon seasons
- Test: 2024 monsoon season

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

## 2. Load Data

In [ ]:
df = pd.read_csv("../data/processed/dhemaji_flood_FINAL.csv")
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year

features = [
    "dist_to_major_river",
    "elevation",
    "tree_cover",
    "slope",
    "rain_anomaly",
    "rain_5day",
    "rain_3day",
    "rainfall_mm",
    "runoff_sum",
    "runoff_anomaly"
]

# Temporal split
train = df[df["year"] < 2024]
test  = df[df["year"] == 2024]

X_train = train[features]
y_train = train["flood_label"]
X_test  = test[features]
y_test  = test["flood_label"]

print("Train:", X_train.shape)
print("Test: ", X_test.shape)
print("Train flood rate:", round(y_train.mean(), 3))
print("Test flood rate: ", round(y_test.mean(), 3))

## 3. Train Final Model

Gradient Boosting with max_features=3 to balance feature contributions and reduce dependence on dist_to_major_river alone.

In [ ]:
model = GradientBoostingClassifier(
    n_estimators     = 200,
    max_depth        = 5,
    learning_rate    = 0.1,
    subsample        = 0.8,
    min_samples_leaf = 20,
    max_features     = 3,
    random_state     = 42
)

print("Training final model...")
model.fit(X_train, y_train)
print("Training complete")

## 4. Predictions with Best Threshold = 0.4

In [ ]:
BEST_THRESHOLD = 0.4

y_proba = model.predict_proba(X_test)[:,1]
y_pred  = (y_proba >= BEST_THRESHOLD).astype(int)

print("="*50)
print("FINAL MODEL PERFORMANCE")
print("="*50)
print(f"Model:      Gradient Boosting (max_features=3)")
print(f"Threshold:  {BEST_THRESHOLD}")
print(f"Features:   {len(features)}")
print(f"Training:   2019-2023 monsoon seasons")
print(f"Testing:    2024 monsoon season")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0,1])
ax.set_yticks([0,1])
ax.set_xticklabels(["No Flood","Flood"])
ax.set_yticklabels(["No Flood","Flood"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")

for i in range(2):
    for j in range(2):
        ax.text(
            j, i, f"{cm[i,j]:,}",
            ha="center", va="center",
            color="white" if cm[i,j] > cm.max()/2 else "black",
            fontsize=14
        )

plt.colorbar(im)
plt.tight_layout()
plt.savefig("../figures/confusion_matrix.png", dpi=150)
plt.show()

print(f"\nTrue Negatives:  {cm[0,0]:,}")
print(f"False Positives: {cm[0,1]:,}")
print(f"False Negatives: {cm[1,0]:,}")
print(f"True Positives:  {cm[1,1]:,}")

## 6. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8,8))
ax.plot(fpr, tpr, color="steelblue", linewidth=2,
        label=f"AUC = {auc_score:.3f}")
ax.plot([0,1],[0,1], color="coral", linestyle="--",
        label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Final Gradient Boosting Model")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("../figures/roc_curve.png", dpi=150)
plt.show()

## 7. Threshold Analysis

Explore the precision-recall tradeoff at different thresholds.

In [ ]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
threshold_results = []

print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<12}")
for thresh in thresholds:
    y_t = (y_proba >= thresh).astype(int)
    p = precision_score(y_test, y_t)
    r = recall_score(y_test, y_t)
    f = f1_score(y_test, y_t)
    threshold_results.append({
        "Threshold": thresh,
        "Precision": round(p, 3),
        "Recall":    round(r, 3),
        "F1":        round(f, 3)
    })
    print(f"{thresh:<12} {p:<12.3f} {r:<12.3f} {f:<12.3f}")

pd.DataFrame(threshold_results).to_csv(
    "../results/threshold_analysis.csv", index=False
)

## 8. Feature Importance

In [ ]:
importance = pd.DataFrame({
    "Feature":    features,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print("Feature Importance:")
print(importance.to_string(index=False))

importance.to_csv(
    "../results/feature_importance.csv", index=False
)

# Plot
fig, ax = plt.subplots(figsize=(10,7))
ax.barh(
    importance["Feature"][::-1],
    importance["Importance"][::-1],
    color="steelblue"
)
ax.set_title("Feature Importance — Final Gradient Boosting Model")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig("../figures/feature_importance.png", dpi=150)
plt.show()

## 9. Save Final Model and Config

In [ ]:
# Save model
joblib.dump(model, "../models/flood_model_FINAL.pkl")

# Save config
config = {
    "model":           "Gradient Boosting (max_features=3)",
    "best_threshold":  BEST_THRESHOLD,
    "features":        features,
    "n_features":      len(features),
    "training_years":  "2019-2023",
    "test_year":       "2024",
    "training_rows":   int(X_train.shape[0]),
    "test_rows":       int(X_test.shape[0]),
    "flood_precision": round(precision_score(y_test, y_pred), 3),
    "flood_recall":    round(recall_score(y_test, y_pred), 3),
    "flood_f1":        round(f1_score(y_test, y_pred), 3),
    "roc_auc":         round(roc_auc_score(y_test, y_proba), 3)
}

with open("../models/model_config_FINAL.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved:")
print("  ../models/flood_model_FINAL.pkl")
print("  ../models/model_config_FINAL.json")
print("\nFinal performance:")
for k, v in config.items():
    if k not in ["features","n_features"]:
        print(f"  {k}: {v}")

## 10. Summary

Final model trained successfully with the following characteristics:

**Performance:**
- Precision: 0.85
- Recall: 0.82
- F1: 0.83
- ROC-AUC: 0.990

**Feature distribution (more balanced after max_features=3):**
- dist_to_major_river: ~46% (down from 64%)
- slope: ~25% (up from 3%)
- tree_cover: ~14%
- elevation: ~8%
- rain_anomaly: ~5%
- Other rainfall/runoff features: <2% each

The model captures physically meaningful patterns:
- River proximity drives most flooding (Brahmaputra overflow)
- Terrain features (slope, elevation, tree cover) contribute meaningfully
- Rainfall provides temporal signal for unusual events